In [ ]:
!pip install streamlit -q
!npm install -g localtunnel

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import re
import math
from collections import Counter
from urllib.parse import urlparse
import scipy.sparse as sp

st.set_page_config(page_title="NEXUS | Advanced Threat Intelligence", page_icon="☣️", layout="wide")

st.markdown("""
    <style>
    /* استيراد خطوط السايبر سكيورتي */
    @import url('https://fonts.googleapis.com/css2?family=Orbitron:wght@500;700&family=Share+Tech+Mono&display=swap');

    /* تغيير الخط الأساسي والخلفية */
    html, body, [class*="css"] {
        font-family: 'Share Tech Mono', monospace !important;
    }
    
    /* إخفاء واجهة Streamlit الافتراضية */
    #MainMenu {visibility: hidden;}
    footer {visibility: hidden;}
    header {visibility: hidden;}

    /* العناوين */
    h1, h2, h3 {
        font-family: 'Orbitron', sans-serif !important;
        color: #00E5FF;
        letter-spacing: 1.5px;
        text-shadow: 0 0 10px rgba(0, 229, 255, 0.3);
    }

    /* صندوق الإدخال */
    .stTextInput>div>div>input {
        background-color: rgba(13, 17, 23, 0.7) !important;
        color: #00E5FF !important;
        border: 1px solid #2d3748 !important;
        border-radius: 8px !important;
        padding: 16px !important;
        font-size: 16px !important;
        transition: all 0.3s ease;
        box-shadow: inset 0 2px 4px rgba(0,0,0,0.5);
    }
    .stTextInput>div>div>input:focus {
        border-color: #00E5FF !important;
        box-shadow: 0 0 15px rgba(0, 229, 255, 0.4), inset 0 2px 4px rgba(0,0,0,0.5) !important;
    }

    /* زر الفحص */
    .stButton>button {
        background: linear-gradient(90deg, #FF003C, #C2002E) !important;
        color: white !important;
        border: none !important;
        border-radius: 8px !important;
        padding: 12px 24px !important;
        font-family: 'Orbitron', sans-serif !important;
        font-size: 16px !important;
        font-weight: bold !important;
        letter-spacing: 2px !important;
        width: 100%;
        transition: all 0.3s ease !important;
        box-shadow: 0 4px 15px rgba(255, 0, 60, 0.3);
    }
    .stButton>button:hover {
        transform: translateY(-2px);
        box-shadow: 0 6px 20px rgba(255, 0, 60, 0.6);
        background: linear-gradient(90deg, #FF1A50, #E60036) !important;
    }

    /* بطاقات الخصائص (Metrics) */
    div[data-testid="stMetric"] {
        background: rgba(22, 27, 34, 0.6);
        border: 1px solid #30363d;
        border-radius: 10px;
        padding: 15px;
        backdrop-filter: blur(10px);
        border-left: 4px solid #00E5FF;
        box-shadow: 0 8px 32px 0 rgba(0, 0, 0, 0.2);
    }
    div[data-testid="stMetricLabel"] {
        color: #8b949e !important;
        font-size: 14px !important;
    }
    div[data-testid="stMetricValue"] {
        color: #ffffff !important;
        font-size: 24px !important;
    }

    /* الخط الفاصل */
    hr {
        border-color: #30363d !important;
        margin: 2rem 0;
    }
    </style>
""", unsafe_allow_html=True)

@st.cache_resource
def load_models():
    model = joblib.load('/kaggle/input/notebooks/promanpro/url-project-ml/XGB_model.pkl')
    tfidf = joblib.load('/kaggle/input/notebooks/promanpro/url-project-ml/tfidf_vectorizer.pkl')
    scaler = joblib.load('/kaggle/input/notebooks/promanpro/url-project-ml/minmax_scaler.pkl')
    encoder = joblib.load('/kaggle/input/notebooks/promanpro/url-project-ml/label_encoder.pkl')
    feature_names = joblib.load('/kaggle/input/notebooks/promanpro/url-project-ml/feature_names.pkl')
    return model, tfidf, scaler, encoder, feature_names

model, tfidf, scaler, encoder, feature_names = load_models()

def calculate_entropy(text):
    if not text: return 0
    entropy = 0
    for x in Counter(text).values():
        p_x = float(x) / len(text)
        entropy -= p_x * math.log(p_x, 2)
    return entropy

def Extract_features(url):
    url = str(url)
    parsed_url = urlparse(url)
    ipv4_pattern = re.compile(r'(([01]?\d\d?|2[0-4]\d|25[0-5])\.){3}([01]?\d\d?|2[0-4]\d|25[0-5])')
    ipv6_pattern = re.compile(r'([0-9a-fA-F]{1,4}:){7,7}[0-9a-fA-F]{1,4}|([0-9a-fA-F]{1,4}:){1,7}:|([0-9a-fA-F]{1,4}:){1,6}:[0-9a-fA-F]{1,4}')
    digit_pattern = re.compile(r'\d')
    letter_pattern = re.compile(r'[a-zA-Z]')
    suspicious_words = ['secure', 'account', 'update', 'login', 'verify', 'bank', 'free', 'bonus', 'admin', 'service', 'client']
    shorteners = ['bit.ly', 'tinyurl.com', 'goo.gl', 't.co', 'ow.ly', 'is.gd', 'buff.ly', 'bit.do']
    
    features = {}
    features['url_length'] = len(url)
    features['hostname_length'] = len(parsed_url.netloc)
    features['path_length'] = len(parsed_url.path)
    features['url_entropy'] = calculate_entropy(url)
    features['count_dots'] = url.count('.')
    features['count_hyphens'] = url.count('-')
    features['count_at'] = url.count('@')
    features['count_question_marks'] = url.count('?')
    features['count_equals'] = url.count('=')
    features['count_ampersands'] = url.count('&')
    features['count_dirs'] = parsed_url.path.count('/')
    features['count_subdomains'] = parsed_url.netloc.count('.')
    features['has_suspicious_words'] = 1 if any(word in url.lower() for word in suspicious_words) else 0
    features['is_shortened'] = 1 if any(short in parsed_url.netloc for short in shorteners) else 0
    features['has_non_standard_port'] = 1 if parsed_url.port not in [None, 80, 443] else 0
    features['has_ip'] = 1 if (ipv4_pattern.search(url) or ipv6_pattern.search(url)) else 0
    features['count_digits'] = len(digit_pattern.findall(url))
    features['count_letters'] = len(letter_pattern.findall(url))
    features['digit_letter_ratio'] = features['count_digits'] / (features['count_letters'] + 1)
    
    return features

st.markdown("<h1>NEXUS // THREAT SCANNER</h1>", unsafe_allow_html=True)
st.markdown("<p style='color: #8b949e; margin-top: -10px;'>XGBoost Powered Malicious URL Detection Engine</p>", unsafe_allow_html=True)

st.markdown("<hr>", unsafe_allow_html=True)

col1, col2 = st.columns([3, 1])

with col1:
    user_url = st.text_input("TARGET URL", placeholder="https://example.com/login-secure", label_visibility="collapsed")

with col2:
    analyze_button = st.button("INITIATE SCAN")

if analyze_button and user_url:
    with st.spinner('Running heuristic analysis...'):
        try:
            features_dict = Extract_features(user_url)
            manual_df = pd.DataFrame([features_dict])
            manual_scaled = scaler.transform(manual_df)
            url_tfidf = tfidf.transform([user_url])
            
            # الدمج والتوقع (بالصيغة الصحيحة للمصفوفة)
            final_features = sp.hstack([manual_scaled, url_tfidf], format='csr')
            prediction_code = model.predict(final_features)[0]
            probabilities = model.predict_proba(final_features)[0]
            
            prediction_label = encoder.inverse_transform([prediction_code])[0]
            confidence = np.max(probabilities) * 100

            st.markdown("<hr>", unsafe_allow_html=True)
            st.markdown("### 📡 SCAN RESULTS")
            
            # كود HTML مخصص لعرض النتيجة بشكل لوحة إنذار احترافية
            if prediction_label == 'benign':
                bg_color = "rgba(0, 255, 136, 0.1)"
                border_color = "#00FF88"
                icon = "✅"
                status = "CLEAN / BENIGN"
            else:
                bg_color = "rgba(255, 0, 60, 0.1)"
                border_color = "#FF003C"
                icon = "☠️"
                status = f"THREAT DETECTED: {prediction_label.upper()}"

            st.markdown(f"""
                <div style="
                    background-color: {bg_color};
                    border: 2px solid {border_color};
                    border-radius: 12px;
                    padding: 20px;
                    text-align: center;
                    box-shadow: 0 0 20px {bg_color};
                    margin-bottom: 30px;">
                    <h2 style="color: {border_color}; margin: 0; font-size: 32px;">{icon} {status}</h2>
                    <p style="color: white; margin-top: 10px; font-size: 18px; font-family: 'Share Tech Mono';">
                        CONFIDENCE LEVEL: <b style="color: {border_color};">{confidence:.2f}%</b>
                    </p>
                </div>
            """, unsafe_allow_html=True)

            st.markdown("### 🔬 EXTRACTED TELEMETRY")
            m1, m2, m3, m4 = st.columns(4)
            m1.metric(label="URL Length", value=features_dict['url_length'])
            m2.metric(label="Entropy", value=f"{features_dict['url_entropy']:.2f}")
            m3.metric(label="Suspicious Words", value="Yes" if features_dict['has_suspicious_words'] else "No")
            m4.metric(label="Direct IP Used", value="Yes" if features_dict['has_ip'] else "No")
            
        except Exception as e:
            st.error(f"SYSTEM ERROR: {e}")

In [ ]:
!pip install pyngrok -q
from pyngrok import ngrok

ngrok.set_auth_token("YOUR_NGROK_TOKEN_HERE") 

public_url = ngrok.connect(8501).public_url
print(f"الرابط المباشر للواجهة: {public_url}")

!streamlit run app.py --server.enableCORS false --server.enableXsrfProtection false